<span style="color:gray"><b>Aprendizaje supervisado >> Análisis de Regresión >> Regresión Lineal</b></span>
<h1 style="margin-top:1px;">Regresión lineal</h1>
<h2>Ejemplo de Regresión Lineal Múltiple: Predicción de precios de propiedades</h2>
<p>Brooklyn, uno de los cinco distritos de la ciudad de Nueva York, ha visto un aumento significativo en los precios inmobiliarios en los últimos años. El aumento de la demanda de viviendas en Brooklyn ha dificultado que los compradores de viviendas puedan predecir el valor de mercado de las propiedades. Esto ha resultado en la necesidad de una herramienta confiable que pueda predecir con precisión el precio de las casas en Brooklyn.</p>
Realizaremos un análisis de regresión lineal múltiple para predecir el precio de las propiedades. 
<h3>Librerías</h3>

In [ ]:
# Tratamiento de datos
# ==============================================================================
import pandas as pd
import numpy as np

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns; sns.set()

# Preprocesado y modelado
# ==============================================================================
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

# Configuración matplotlib
# ==============================================================================
plt.rcParams['image.cmap'] = "bwr"
#plt.rcParams['figure.dpi'] = "100"
plt.rcParams['savefig.bbox'] = "tight"
style.use('ggplot') or plt.style.use('ggplot')
plt.style.use('tableau-colorblind10')
sns.set_style('whitegrid')

# Configuración warnings
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

Configuramos el entorno para que podamos ver todo el marco de datos.

In [ ]:
pd.set_option('display.max_columns', 111)

<h3>Datos</h3>
Puede descargar el conjunto de datos del siguiente enlace:<br/>
<a href="https://www.kaggle.com/datasets/tianhwu/brooklynhomes2003to2017?select=brooklyn_sales_map.csv">dataset</a>

In [ ]:
HP=pd.read_csv('brooklyn_sales_map.csv')
HP.head()

In [ ]:
HP.shape

Describimos los datos y agregamos en ellos el número de NaN en cada columna en la última fila. Vemos que hay muchas columnas con NaNs. Nuestro próximo paso será limpiar estos datos tanto como sea posible.

In [ ]:
HP.describe().append(HP.isnull().sum().rename('isnull'))

In [ ]:
HP.columns.values

<h4>Analicemos la distribución de los precios</h4>

In [ ]:
plt.hist(HP['sale_price'])

In [ ]:
plt.ylim(0,1500000)
plt.boxplot(HP['sale_price'])

No es lógico que el del precio de venta mínimo sea $0, de hecho por la región ignoraremos todas las propiedades que tienen un precio menor a 100000 y aunque el diagrama de cajas marca demasiados outliers las propiedades si llegan a tener esos precios por lo que solo quitaremos las que tienen un valor superior a 1.3 millones

In [ ]:
mask = (HP['sale_price'] > 100000)& (HP['sale_price'] < 1300000)
HP = HP[mask]

In [ ]:
plt.boxplot(HP['sale_price'])

In [ ]:
plt.hist(HP['sale_price'])

In [ ]:
HP.shape

<h3>Selección de características e Ingeniería de características</h3>

El dataset consiste en dos conjuntos de datos concatenados de dos fuentes que se dejaron unidos para formar el conjunto de datos final tal como está disponible. Uno de los datos originales es del <span style="color:green">Departamento de Finanzas de la Ciudad de Nueva York</span> (datos de ventas de viviendas) y el otro del <span style="color:green">Departamento de Planificación Urbana de la Ciudad de Nueva York</span> (<b>PLUTO y MapPLUTO</b>).<br/>

<h4>Descartando algunas columnas irrelevantes</h4>
Se puede hacer una gran simplificación eliminando las columnas duplicadas. También hay algunas columnas que hacen referencia a varios ID de mapa o números de archivo que supuestamente no tienen impacto en ninguna de las predicciones realizadas.
<ul>
    <li><b>Unnamed: 0</b> número de registro. Irrelevante</li>
    <li>Eliminamos tanto <b>borough</b> como <b>Borough</b> ya que estamos trabajando solo en el municipio de Brooklyn.</li>
    <li><b>AreaSource</b> Un código que indica el archivo de origen que se utilizó para determinar el ÁREA TOTAL DE PLANTA DE CONSTRUCCIÓN del lote fiscal (BldgArea)</li>
    <li><b>Block, XCoord y YCoord</b>: estos detalles no agregarán mucho valor ya que ya tenemos información sobre el vecindario (neighborhood). Agregar estas características solo hará que el modelo sea innecesariamente complejo.</li>
    <li><b>LotArea</b> Área total del lote fiscal, expresada en pies cuadrados redondeados al número entero más cercano. Esto es lo mismo que <b>lot</b></li>
    <li><b>BldgArea</b> El área bruta total en pies cuadrados. Igual que <b>gross_sqft</b></li>
    <li><b>BldgClass</b> misma información que <b>building_class</b></li>
    <li><b>Easements,easement</b>, 'servidumbre'. El número de servidumbres en el lote fiscal, esto no es importante para la mayoría de las ventas.</li>
    <li><b>sale_date</b> fecha de venta pero mantenemos solo el año en <b>year_of_sale</b></li>
    <li><b>ZipCode</b> misma información que <b>zip_code</b></li>
    <li><b>CT2010, CB2010 y Tract2010</b>: variables categóricas que están asociadas a la ubicación de la vivienda y, por tanto, deben ser excluidas.</li>
    <li><b>ZoneMap</b> El número de mapa de zonificación del Departamento de Planificación Urbana asociado con las coordenadas X e Y del lote fiscal.</li>
    <li><b>Sanborn</b> El número de mapa de Sanborn Map Company asociado con el bloque fiscal y el <b>lote</b>.</li>
    <li><b>Health Cent, SchoolDist</b>: estas también son variables categóricas que están enmascaradas con números y asociadas con la ubicación y, por lo tanto, se pueden eliminar.</li>
    <li><b>TaxMap</b> El número de volumen del mapa de impuestos en papel del Departamento de Finanzas asociado con el bloque fiscal y el <b>lote</b>.</li>
    <li><b>YearAlter1 y YearAlter2</b>: variable asociada a la edad del edificio, por lo que se descarta</li>
    <li><b>'CondoNo'</b> El número de condominio asignado al complejo, irrelevante.</li>
    <li><b>'APPBBL'</b> El Municipio, Bloque Fiscal y Lote Fiscal de origen de la propiedad antes de la fusión, división o conversión de la propiedad a un condominio. El Apportionment BBL solo está disponible para fusiones, escisiones y conversiones desde 1984.</li>
    <li><b>PLUTOMapID:</b> Un código que indica si el lote fiscal está en el archivo PLUTO y/o el DTM modificado y/o el DTM modificado recortado en el archivo Shoreline.</li>
    <li><b>MAPPLUTO_F</b>: sin descripción.</li>
    <li><b>Versión</b> El número de versión relacionado con el lanzamiento de PLUTO.</li>
    <li><b>land_sqft</b> es la misma columna que LotArea</li>
    <li><b>Address</b> misma información que <b>address</b></li>
    <li><b>UnitsRes, UnitsTotal</b> tienen la misma información que <b>residential_units y commercial_units</b> respectivamente</li>
    <li>Para este análisis basta con saber cuantas unidades son residencias y cuantas son comerciales sin entrar en detalle de que tipo de comercio se trata, por lo que eliminaremos las columnas: <b>ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource</b></li>
</ul>

In [ ]:
HP.drop(columns=['Unnamed: 0','borough','Borough','AreaSource','block','LotArea','XCoord','YCoord','BldgArea',
                 'BldgClass','Easements','easement','sale_date','ZipCode','CT2010', 'CB2010','Tract2010',
                 'Sanborn','HealthCent','ZoneMap','SchoolDist','TaxMap','YearAlter1','YearAlter2','CondoNo','APPBBL',
                 'PLUTOMapID','MAPPLUTO_F','Version','land_sqft','Address','UnitsRes','UnitsTotal','ComArea','ResArea', 
                 'OfficeArea','RetailArea','GarageArea', 'StrgeArea', 'FactryArea', 'OtherArea'], axis=1, inplace=True)
HP.shape

<h4>Datos Faltantes</h4>

In [ ]:
#% Porcentaje de datos faltantes
infonan=HP.isna().sum()/len(HP)*100
infonan

Parece que hay una cantidad sustancial de valores faltantes en la mayoría de las columnas. Para solucionar este problema, he establecido un valor límite del 25 % para el valor faltante. Las columnas que excedan este umbral serán eliminadas.

In [ ]:
HP.dropna(thresh=0.75*len(HP),axis=1,inplace=True)

In [ ]:
print(HP.columns.values)
HP.shape

<h4>Cardinalidad</h4>
Otro problema que enfrentamos es la <a href="https://es.wikipedia.org/wiki/Cardinalidad">cardinalidad</a> en nuestros datos categóricos. Las columnas de alta cardinalidad, con un gran número de categorías, pueden aumentar la complejidad de nuestro modelo. Para mitigar esto, eliminaremos columnas de alta cardinalidad, con la excepción de Neighborhood (barrio), Building Class(clase de edificio) y tax_class. Estas dos columnas tienen un alto impacto potencial en nuestra variable objetivo y se conservarán. Para reducir aún más la cardinalidad de estas columnas, deberemos considerar estrategias apropiadas, como agrupar las categorías minoritarias.

In [ ]:
cat_cols = HP.select_dtypes('object').columns
k = 0
cardinality_cols = []

for col in cat_cols:
    value_counts = HP[col].value_counts(normalize=True).round(2)
    if len(value_counts) > 5 or len(value_counts) < 2:
        cardinality_cols.append(col)
    else:
        print("====================================================================")
        print(col)
        print(value_counts)

cardinality_cols.remove('neighborhood')
cardinality_cols.remove('building_class_at_sale')
cardinality_cols.remove('tax_class')
print(f"Removed Columns: {cardinality_cols}")
HP.drop(columns=cardinality_cols, axis=1, inplace=True)

In [ ]:
HP['neighborhood'].value_counts(normalize=True).round(2)

<h5>Reducción de Cardinalidad</h5>
Hemos observado que la columna "barrio" tiene una cantidad excesiva de categorías, con 61 en total. Para simplificar los datos, agruparemos las categorías con menos del 1% de representación en una categoría "OTROS". El mismo proceso se puede aplicar a las columnas "building_class_at_sale" y "commercial_units".

In [ ]:
neighborhood = HP['neighborhood'].value_counts(normalize=True).round(2)
idx = np.where(neighborhood > 0.01)[0].max()
places = neighborhood.index[:idx]
HP['neighborhood'] =HP['neighborhood'].apply(lambda x: x if x in places else 'OTHER')
HP['neighborhood'].value_counts(normalize=True)

In [ ]:
building_class = HP['building_class_at_sale'].value_counts(normalize=True).round(2)
idx = np.where(building_class > 0.01)[0].max()
places = building_class.index[:idx]
HP['building_class_at_sale'] =HP['building_class_at_sale'].apply(lambda x: x if x in places else 'OTHER')
HP['building_class_at_sale'].value_counts(normalize=True)

<H5>Descartar columnas de varianza baja</H5>
Otro aspecto que debemos abordar es la presencia de columnas de baja varianza. Estas columnas no tienen mucha variación en sus valores y por lo tanto no aportan mucho valor al modelo. Se recomienda eliminar dichas columnas para que el modelo sea más preciso.

In [ ]:
num_cols = HP.select_dtypes('number').columns

low_var_cols = []
for col in num_cols:
    scaled = (HP[col] - HP[col].mean()) / HP[col].std()
    variance = scaled.var().round(4)
    if variance == 0 or HP[col].std() == 0:
        low_var_cols.append(col)
#     else:
#         print(col, variance)

HP.drop(columns=low_var_cols, axis=1, inplace=True)
print(f"Removed Columns: {low_var_cols}")
print(HP.shape, len(low_var_cols))

<H5>Extraer características nuevas y eliminar las que no son útiles</H5>
Cuando se trata de determinar el valor de una casa, la edad de la propiedad es un factor crucial. En lugar de usar el año de construcción, podemos calcular la edad de la casa tomando el promedio de las columnas "year_built" y "YearBuilt", y restando el resultado de 2023.
<p>Algunas columnas como "zip_code" y "year_of_sale" pueden no tener mucho impacto en el valor de la casa y pueden eliminarse. Además, después de calcular la edad, también se pueden eliminar las columnas "year_built" y "YearBuilt".</p>

In [ ]:
HP['age'] = 2023 - (HP['year_built'] + HP['YearBuilt']) // 2
HP.drop(['zip_code', 'year_built', 'YearBuilt', 'year_of_sale'], axis=1, inplace=True)

<h3>Lidiando con la multicolinealidad</h3>
<h4>Usar matriz de correlación</h4>
Reducir la cantidad de características mediante la eliminación de características independientes correlacionadas puede ayudar a prevenir la <span style="color:red">multicolinealidad</span>. Esto se debe a que las características altamente correlacionadas no aportan ninguna información nueva a la mesa.

In [ ]:
corr = HP.drop('sale_price', axis=1).corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(np.bool))
drop = [column for column in upper.columns if any(np.abs(upper[column]) > 0.7)]
print(f"columns dropped are: {drop}")
HP.drop(columns=drop, axis=1, inplace=True)

In [ ]:
print(HP.columns.values)
HP.shape

<h3>Columnas categoricas a numéricas</h3>
El método <span style="color:blue">astype('category')</span> trata a la columna como una propiedad que tiene categorias, asociándole un valor numérico a cada una (0,1,2...)
df[colum].<span style="color:blue">cat.codes</span> es el atributo de la columna que tiene el número de categoria asociado.

In [ ]:
HP['neighborhood']=HP['neighborhood'].astype('category')
HP['neighborhood']=HP['neighborhood'].cat.codes

<h3>Manejo de valores faltantes</h3>

In [ ]:
print(HP['SplitZone'].unique())
HP['SplitZone']= HP['SplitZone'].fillna(value='N')
HP['SplitZone']= HP['SplitZone'].astype('category')
HP['SplitZone']= HP['SplitZone'].cat.codes
print(HP['SplitZone'].unique())

In [ ]:
print(HP['tax_class'].unique())
print(HP['tax_class'].isnull().sum())
print(HP['tax_class_at_sale'].unique())
print(HP['tax_class_at_sale'].isnull().sum())
HP['tax_class'] = HP['tax_class'].map({'1B': 5, '2A': 6, '2B':7, '1A':8, '2C':9, '3':3,'4':4,'2':2,'1':1})
HP['tax_class'].fillna(HP['tax_class_at_sale'], inplace=True)
del HP['tax_class_at_sale']

Aunque ProxCode, BstmCode son numéricos conviene remplazarlos por la moda

In [ ]:
HP['ProxCode']= HP['ProxCode'].fillna(HP['ProxCode'].mode()[0])
HP['BsmtCode']= HP['BsmtCode'].fillna(HP['BsmtCode'].mode()[0])

In [ ]:
numeric = HP.select_dtypes(include=np.number)
numeric_columns = numeric.columns
HP[numeric_columns] = HP[numeric_columns].fillna(HP.median(numeric_only= True))

In [ ]:
HP.describe().append(HP.isnull().sum().rename('isnull'))

In [ ]:
HP['IrrLotCode'].unique()
HP['IrrLotCode']= HP['IrrLotCode'].fillna(HP['IrrLotCode'].mode()[0])
HP['IrrLotCode']= HP['IrrLotCode'].astype('category')
#HP['IrrLotCode']= HP['IrrLotCode'].cat.codes
HP['building_class_at_sale']=HP['building_class_at_sale'].astype('category')

cat_columns = HP.select_dtypes(['category']).columns
print(cat_columns)
HP[cat_columns] = HP[cat_columns].apply(lambda x: x.cat.codes)

In [ ]:
HP.isnull().sum().sum()

<h3>Manejo de la multicolinealidad</h3>
<h4>Prueba VIF</h4>
A continuación, eliminamos las variables que están altamente correlacionadas mediante la evaluación de su <span style="color:red">factor de inflación de la varianza</span>. Eliminamos una variable en cada paso y nuevamente calculamos el VIF y luego eliminamos la siguiente. Eliminamos todas las variables con VIF>5 de esta manera para minimizar la correlación entre las variables.<br/>
Este test solo puede aplicarse a columnas numéricas<br/>
Consulte más sobre esta prueba en <a href="https://www.geeksforgeeks.org/detecting-multicollinearity-with-vif-python/">test VIF</a>

In [ ]:
X = HP.drop(['sale_price'],axis=1)

In [ ]:
X.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

Eliminamos la columna con el mayor VIF que sea mayor a 5.0, que en este caso es SanitBoro con 5050.2 y repetimos el proceso hasta que todos los VIF sean menores o iguales a 5.0<br/>
<b>*NOTA:</b> Elimine solo 1 columna a la vez, ya que el valor varia al ir descartando características

In [ ]:
del HP['SanitBoro']
X = HP.drop('sale_price',axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['CD']
X = HP.drop('sale_price',axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['LotType']
X = HP.drop('sale_price',axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

No eliminaremos a building_class_at_sale pues como dijimos antes es de gran valor para el modelo

In [ ]:
#del HP['building_class_at_sale']
X = HP.drop(['sale_price','building_class_at_sale'],axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['LotDepth']
X = HP.drop(['sale_price','building_class_at_sale'],axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['LandUse']
X = HP.drop(['sale_price','building_class_at_sale'],axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['ResidFAR']
X = HP.drop(['sale_price','building_class_at_sale'],axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del HP['BsmtCode']
X = HP.drop(['sale_price','building_class_at_sale'],axis=1)
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

Una vez que todos los valores VIF estan por debajo de 5 consultamos las columnas con las que trabajaremos

In [ ]:
print(HP.columns)
HP.shape

In [ ]:
HP.head()

<h3>Visualización de datos</h3>
Visualizamos la distribución respecto de la variable sale_price

In [ ]:
viz_df = HP.copy()
median_home_price = viz_df['sale_price'].median().round(2)
print(median_home_price)
k = 0
plt.figure(figsize=(20, 24))
for col in viz_df.drop('sale_price', axis=1).columns:
    plt.subplot(6, 3, k + 1)
    if viz_df[col].dtype == 'float64' or viz_df[col].dtype =='int':
        sns.scatterplot(data=viz_df, x=col, y='sale_price')
        plt.title(f"Sale Price Vs {col}")
    else:
        group = viz_df[[col, 'sale_price']].groupby(by=col).median()
        sns.barplot(data=group, x=group.index, y='sale_price')
        plt.axhline(y=median_home_price, label='Median Home Price', color='black', linestyle='--', linewidth=0.7)
        plt.title(f"Avg Home Price by {col}")
        plt.legend()
    k += 1
plt.tight_layout()

Parece que las columnas numéricas no tienen una fuerte correlación lineal con la columna de destino, lo que dificulta predecir el precio de venta usando solo estas columnas.

<p>Pero, parece que las casas con lotes regulares tienden a tener un precio de venta más alto en comparación con aquellas con lotes irregulares. De manera similar, las casas que no están ubicadas en zonas divididas tienden a ser más valiosas.</p>

<h3>Análisis de correlación</h3>

In [ ]:
# Correlación entre columnas numéricas
# ==============================================================================

def tidy_corr_matrix(corr_mat):
    '''
    Función para convertir una matriz de correlación de pandas en formato tidy
    '''
    corr_mat = corr_mat.stack().reset_index()
    corr_mat.columns = ['variable_1','variable_2','r']
    corr_mat = corr_mat.loc[corr_mat['variable_1'] != corr_mat['variable_2'], :]
    corr_mat['abs_r'] = np.abs(corr_mat['r'])
    corr_mat = corr_mat.sort_values('abs_r', ascending=False)
    
    return(corr_mat)

corr_matrix = HP.select_dtypes(include=['float64', 'int']).corr(method='pearson')
tidy_corr_matrix(corr_matrix).head(10)

In [ ]:
# Heatmap matriz de correlaciones
# ==============================================================================
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9, 9))

sns.heatmap(
    corr_matrix,
    annot     = True,
    cbar      = False,
    annot_kws = {"size": 8},
    vmin      = -1,
    vmax      = 1,
    center    = 0,
    cmap      = sns.diverging_palette(20, 220, n=200),
    square    = True,
    ax        = ax
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation = 45,
    horizontalalignment = 'right',
)

ax.tick_params(labelsize = 10)
plt.show()

<h3>Comprendiendo las características con las que trabajaremos</h3>
<table>
    <tr><th>Característica</th><th>Descripción</th></tr>
    <tr><td><b>sale_price</b></td><td>precio de venta de la propiedad (Objetivo)</td></tr>
    <tr><td><b>neighborhood</b></td><td>barrio de la propiedad.</td></tr>
    <tr><td><b>tax_class</b></td><td>clase fiscal de la propiedad (impuestos)</td></tr>
    <tr><td><b>lote</b></td><td>Área total del lote fiscal, expresada en pies cuadrados redondeados al entero más cercano.</td></tr>
    <tr><td><b>residential_units</b></td><td>Unidades Residenciales (casas, departamentos)<br/>
        La suma de unidades residenciales en todos los edificios en el lote fiscal.</td></tr>
    <tr><td><b>commercial_units</b></td><td>Unidades comerciales (oficinas, ventas, etc.)</td></tr>
    <tr><td><b>SplitZone</b></td><td>Indicador de límite dividido<br/>
        Un código que indica si el lote fiscal se divide entre varias características de zonificación.<br/>
        El indicador de límite dividido es igual a “Y” si el lote fiscal tiene un valor para distrito de zonificación 2, <br/>
        Superposición comercial 2 o Límite de distrito especial 2.</td></tr>
    <tr><td><b>NumBldgs</b></td><td>Número de edificios en el lote fiscal.</td></tr>
    <tr><td><b>NumFloors</b></td><td>Número de pisos de la propiedad</td></tr>
    <tr><td><b>BldgFront BldgDepth</b></td><td>Medidas de la propiedad frente y profundidad.</td></tr>
    <tr><td><b>ProxCode</b></td><td>Código de proximidad<br/>
        Un código que describe la relación física del edificio con los edificios vecinos.
        <ol>
            <li value="0">No disponible</li>
            <li>Independiente</li>
            <li>Semiadjunto</li>
            <li>Adjunto</li>
        </ol></td></tr>
    <tr><td><b>IrrLotCode</b></td><td>Código que indica si el lote fiscal tiene una forma irregular(Y) o no (N).</td></tr>
    <tr><td><b>Age</b></td><td>Edad de la propiedad, calculada a partir del año de construcción (yearbuild)</td></tr>
</table>

In [ ]:
# Gráfico de distribución para cada variable numérica
# ==============================================================================
# Ajustar número de subplots en función del número de columnas
fig, axes = plt.subplots(nrows=6, ncols=3, figsize=(9, 16))
axes = axes.flat
columnas_numeric = HP.select_dtypes(include=['float64', 'int','int8']).columns

for i, colum in enumerate(columnas_numeric):
    sns.histplot(data=HP,x=colum,stat= "count",kde=True,
        color   = (list(plt.rcParams['axes.prop_cycle'])*2)[i%8]["color"],
        line_kws= {'linewidth': 2},alpha =0.3,ax=axes[i]
    )
    axes[i].set_title(colum, fontsize = 10, fontweight = "bold")
    axes[i].tick_params(labelsize = 8)
    axes[i].set_xlabel("")

fig.tight_layout()
plt.subplots_adjust(top = 0.9)
fig.suptitle('Distribución variables numéricas', fontsize = 10, fontweight = "bold");
plt.show()

In [ ]:
X = HP.drop('sale_price',axis=1)
Y = HP['sale_price']
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X,Y.values.reshape(-1,1),test_size=0.3, random_state=42)

In [ ]:
# Creación del modelo utilizando el modo fórmula (similar a R)
# ==============================================================================
datos_train = pd.DataFrame(
                     np.hstack((Xtrain, Ytrain)),
                     columns=np.append(Xtrain.columns.values,'sale_price')
               )
modelo = smf.ols(formula = 'sale_price ~ neighborhood+building_class_at_sale+tax_class+lot+residential_units+commercial_units+SplitZone+'+
               'NumBldgs+NumFloors+LotFront+BldgFront+BldgDepth+ProxCode+IrrLotCode+CommFAR+age', data = datos_train)
modelo = modelo.fit()
print(modelo.summary())

Podriamos descartar a NumBldgs y lot por tener un p-value mayor a 0.01 pero esperemos a ver como se comporta el modelo utilizando las variables categóricas.

<h3>Variables categóricas como predictores</h3>
<p>Cuando se introduce una variable categórica como predictor, un nivel se considera el de referencia (normalmente codificado como 0) y el resto de niveles se comparan con él. En el caso de que el predictor categórico tenga más de dos niveles, se generan lo que se conoce como variables <b>dummy</b> o <span style="color:red">one-hot-encodding</span>, que son variables creadas para cada uno de los niveles del predictor categórico y que pueden tomar el valor de 0 o 1. Cada vez que se emplee el modelo para predecir un valor, solo una variable dummy por predictor adquiere el valor 1 (la que coincida con el valor que adquiere el predictor en ese caso) mientras que el resto se consideran 0. El valor del coeficiente parcial de regresión $\beta_j$ de cada variable dummy indica el porcentaje promedio en el que influye dicho nivel sobre la variable dependiente y en comparación con el nivel de referencia de dicho predictor.</p>

En nuestro modelo la variable respuesta $sale\_price$ se predice en función de: <i>neighborhood, tax_class, residential_units, commercial_units, land_sqft, SplitZone, LotArea, ComArea, OfficeArea, RetailArea, StrgeArea, FactryArea, OtherArea, NumBldgs, BldgFront, BldgDepth, ProxCode, CommFAR, age</i> de la propiedad. Las variables <span style="color:purple">neighborhood, tax_class, commercial\_units, SplitZone, ProxCode e IrrLotCode</span> son cualitativas con 25, 9, 2, 4 y 2 niveles respectivamente. Para cada característica se crea una variable nueva por cada nivel, cuyo valor puede ser 0 o 1. De tal forma que la ecuación del modelo completo es:
$$sale\_price=\beta_0+\beta_1 neighborhood_1+\beta_2 neighborhood_2+\beta_3 neighborhood_3+... $$
$$+\beta_{25} neighborhood_{25}+\beta_{26} tax\_class_2+\beta_{27} tax\_class_3+\beta_{28} tax\_class_4+...+\beta_{33} tax\_class_9$$
$$+\beta_{34} lot +\beta_{35} residential\_units+\beta_{36} commercial\_units+\beta_{37} SplitZone_1+\beta_{38} NumFloors$$
$$+\beta_{39}LotFront+\beta_{40}NumBldgs +\beta_{41} BldgFront+\beta_{42} BldgDepth+\beta_{43} ProxCode_2$$
$$+\beta_{44} ProxCode_3+\beta_{45} ProxCode_4+\beta_{46} IrrLotCode_1 +\beta_{47} CommFAR+\beta_{48} age$$

Si la propiedad tiene neighborhood=15, tax_class=5, SplitZone=1, ProxCode=2 e IrrLoteCode=0 el resto de las variables dummy se consideran 0, de forma que el modelo para este caso se queda en:
$$sale\_price=\beta_0+\beta_{15} neighborhood_{15}+\beta_{29} tax\_class_5+\beta_{34} lot $$
$$+\beta_{35} residential\_units+\beta_{36} commercial\_units+\beta_{37} SplitZone_1$$
$$+\beta_{38} NumFloors+\beta_{39}LotFront+\beta_{40}NumBldgs +\beta_{41} BldgFront$$
$$+\beta_{42} BldgDepth+\beta_{43} ProxCode_2+\beta_{47} CommFAR+\beta_{48} age$$

Se toman como predictores categóricos a neighborhood, SplitZone, ProxCode, tax_class e IrrLotCode

In [ ]:
datos_train = pd.DataFrame(
                     np.hstack((Xtrain, Ytrain)),
                     columns=np.append(Xtrain.columns.values,'sale_price')
               )
modelo = smf.ols(formula = 'sale_price ~ C(neighborhood)+C(building_class_at_sale)+C(tax_class)+lot+residential_units+commercial_units+C(SplitZone)+'+
               'NumBldgs+NumFloors+LotFront+BldgFront+BldgDepth+C(ProxCode)+C(IrrLotCode)+CommFAR+age', data = datos_train)
modelo = modelo.fit()
print(modelo.summary())

Ahora si podemos eliminar a residential_units, LotFront y CommFAR por ser insignificante para el modelo

<h3>Interacción entre predictores</h3>
El modelo lineal a partir del cual se han obtenido las conclusiones asume que el efecto sobre el precio de venta debido a un incremento en alguna característica es independiente del incremento en las otras. Por ejemplo, el modelo lineal considera que el efecto promedio sobre el precio de venta debido a aumentar en una unidad el número de unidades residenciales es siempre de 114.3, independientemente de la cantidad del resto de características. Sin embargo, esto no tiene por qué ser necesariamente así, puede existir interacción entre los predictores de forma que, el efecto de cada uno de ellos sobre la variable respuesta, depende en cierta medida del valor que tome el otro predictor.
<p>Tal y como se ha definido previamente, un modelo lineal con dos predictores sigue la ecuación:</p>
$$y=\beta_0+\beta_1 x_1+\beta_2 x_2+\epsilon$$
Acorde a esta definición, el incremento de una unidad en el predictor $x_1$ produce un incremento promedio de la variable y de $\beta_1$. Modificaciones en el predictor $x_2$ no alteran este hecho, y lo mismo ocurre con  $x_2$ respecto a $x_1$. Para que el modelo pueda contemplar la interacción entre ambos, se introduce un tercer predictor, llamado <span style="color:red">interaction term</span>, que se construye con el producto de los predictores $x_1$ y $x_2$.
$$y=\beta_0+\beta_1 x_1+\beta_2 x_2+\beta_3 x_1 x_2+e$$
La reorganización de los términos resulta en:
$$y=\beta_0+(\beta_1+\beta_3 x_2)x_1+\beta_2 x_2+e$$
El efecto de $x_1$ sobre y ya no es constante, sino que depende del valor que tome $x_2$.
<p>En los modelos de regresión lineal múltiple que incorporan interacciones entre predictores hay que tener en cuenta el <span style="color:red">hierarchical principle</span>, según el cual, si se incorpora al modelo una interacción entre predictores, se deben incluir siempre los predictores individuales que participan en la interacción, independientemente de que su p-value sea significativo o no.</p>
<p>En stastmodels se puede introducir interacción entre predictores de dos formas:
<ul>
    <li>Indicandolo en la fórmula con el término *: sale_price ~ NumFloors * BldgFront</li>
    <li>Añadiendo una nueva columna con la multiplicación de los predictores cuya interacción se quiere incluir en el modelo.</li>
</ul>
</p>

In [ ]:
datos_train = pd.DataFrame(
                     np.hstack((Xtrain, Ytrain)),
                     columns=np.append(Xtrain.columns.values,'sale_price')
               )
#+C(RetailArea)
modeloint = smf.ols(formula = 'sale_price ~ C(neighborhood)+C(building_class_at_sale)+C(tax_class)+lot+commercial_units+C(SplitZone)+'+
               'NumBldgs+NumFloors+BldgFront+BldgDepth+C(ProxCode)+C(IrrLotCode)+age+NumFloors*BldgFront',
                 data = datos_train)
modeloint = modeloint.fit()
print(modeloint.summary())

Los resultados muestran una evidencia clara de que la interacción NumFloors BldgFront es significativa (p-value* muy próximo a 0).
<p>El modelo que incorpora la interacción tiene un Adjusted R-squared de 0.253, un valor superior al 0.252 del modelo que solo contemplaba el efecto de los predictores por separado. Ahora bien, ¿Es suficiente esta diferencia para afirmar que el modelo con interacción es superior? Una forma de responder a esta pregunta es recurriendo al F-test (<span style="color:red"><b>ANOVA</b></span>).</p>

<h3>Comparación de modelos mediante test F-test (ANOVA)</h3>
Supóngase un modelo M y otro modelo m, de menor tamaño, formado por un subconjunto de los predictores contenidos en M. Si la diferencia en el ajuste es muy pequeña, acorde al principio de parsimonia, el modelo m es más adecuado. Es posible contrastar si la diferencia en ajuste es significativa mediante la comparación de los residuos. En concreto el estadístico empleado es:
$$\frac{RSS_m−RSS_M}{RSS_M}$$
Para evitar que el tamaño del modelo influya en el contraste, se divide la suma de residuos cuadrados RSS de cada modelo entre sus grados de libertad. El estadístico resultante sigue una distribución F.
$$\frac{(RSS_m−RSS_M)/(df_m−df_M)}{(RSS_M/df_M)}~F_{df_m−df_M, df_M}$$
donde df son los grados de libertad del modelo, que equivalen al número de observaciones menos el número de predictores.
<p>En los apartados anteriores, se entrenó un modelo que incluía interacciones y otro que no. Se procede a realizar un test de hipótesis que evalúe la hipótesis nula de que ambos modelos se ajustan a los datos igual de bien. Es decir, que permita determinar si un modelo es mejor que el otro prediciendo la variable respuesta. Esto puede hacerse con la función <span style="color:purple">anova_lm</span>() de statsmodels.stats.anova.</p>

In [ ]:
anova_lm(modelo, modeloint)

El test ANOVA no encuentra evidencias claras (p-value de NaN) de que el modelo que incluye la interacción entre predictores sea capaz de modelar mejor la variable respuesta.
<p>Esta forma de comparar modelos está muy arraigada en la comunidad estadística. En la comunidad de machine learning es mucho más común comparar modelos con técnicas de validación como la <span style="color:red">validación cruzada</span>.</p>

In [ ]:
# Se eliminan las columnas del conjunto de train y test
columns = ['residential_units', 'LotFront','CommFAR']
Xtrain = Xtrain.drop(columns=columns)
Xtest  = Xtest.drop(columns =columns)

In [ ]:
modelo = smf.ols(formula = 'sale_price ~ C(neighborhood)+C(building_class_at_sale)+C(tax_class)+lot+commercial_units+C(SplitZone)+'+
               'NumBldgs+NumFloors+BldgFront+BldgDepth+C(ProxCode)+C(IrrLotCode)+age',
                 data = datos_train)
modelo = modelo.fit()
print(modelo.summary())

Mantener el vecindario en las características de venta en el modelo fue una buena elección, ya que desempeñó un papel importante en la determinación del precio de venta. No reducir la cardinalidad de estas características puede haber llevado a mejores puntajes, pero habría tenido el costo de una mayor complejidad.

In [ ]:
# Diagnóstico errores (residuos) de las predicciones de entrenamiento
# ==============================================================================
Ytrain = Ytrain.flatten()
prediccion_train = modelo.predict(exog = Xtrain)
residuos_train   = prediccion_train - Ytrain

In [ ]:
# Gráficos
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(8, 14))

axes[0, 0].scatter(Ytrain, prediccion_train, edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 0].plot([Ytrain.min(), Ytrain.max()], [Ytrain.min(), Ytrain.max()],
                'k--', color = 'black', lw=2)
axes[0, 0].set_title('Valor predicho vs valor real', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Predicción')
axes[0, 0].tick_params(labelsize = 7)

axes[0, 1].scatter(list(range(len(Ytrain))), residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 1].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[0, 1].set_title('Residuos del modelo', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residuo')
axes[0, 1].tick_params(labelsize = 7)

sns.histplot(
    data    = residuos_train,
    stat    = "density",
    kde     = True,
    line_kws= {'linewidth': 1},
    color   = "firebrick",
    alpha   = 0.3,
    ax      = axes[1, 0]
)

axes[1, 0].set_title('Distribución residuos del modelo', fontsize = 10,
                     fontweight = "bold")
axes[1, 0].set_xlabel("Residuo")
axes[1, 0].tick_params(labelsize = 7)


sm.qqplot(
    residuos_train,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1], 
    color = 'firebrick',
    alpha = 0.4,
    lw    = 2
)
axes[1, 1].set_title('Q-Q residuos del modelo', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

axes[2, 0].scatter(prediccion_train, residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[2, 0].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[2, 0].set_title('Residuos del modelo vs predicción', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Predicción')
axes[2, 0].set_ylabel('Residuo')
axes[2, 0].tick_params(labelsize = 7)

# Se eliminan los axes vacíos
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Diagnóstico residuos', fontsize = 12, fontweight = "bold");
plt.show()

<h3>Test de normalidad</h3>
Se comprueba si los residuos siguen una distribución normal empleando dos test estadísticos: Shapiro-Wilk test y D'Agostino's K-squared test. Este último es el que incluye el summary de statsmodels bajo el nombre de Omnibus.

<p>En ambos test, la hipótesis nula considera que los datos siguen una distribución normal, por lo tanto, si el p-value no es inferior al nivel de referencia alpha seleccionado, no hay evidencias para descartar que los datos se distribuyen de forma normal.</p>

In [ ]:
# Normalidad de los residuos Shapiro-Wilk test
# ==============================================================================
shapiro_test = stats.shapiro(residuos_train)
shapiro_test

In [ ]:
# Normalidad de los residuos D'Agostino's K-squared test
# ==============================================================================
k2, p_value = stats.normaltest(residuos_train)
print(f"Estadítico= {k2}, p-value = {p_value}")

Ambos test muestran claras evidencias para rechazar la hipótesis de que los datos se distribuyen de forma normal (p-value << 0.01).

<h3>Predicciones</h3>
Una vez entrenado el modelo, se pueden obtener predicciones para nuevos datos. Los modelos de statsmodels permiten calcular los intervalos de confianza asociados a cada predicción.

In [ ]:
# Predicciones con intervalo de confianza 
# ==============================================================================
predicciones = modelo.get_prediction(exog = Xtrain).summary_frame(alpha=0.05)
predicciones.head(5)

In [ ]:
# Error de test del modelo 
# ==============================================================================
Xtest = sm.add_constant(Xtest, prepend=True)
predicciones = modelo.predict(exog = Xtest)
rmse = mean_squared_error(
        y_true  = Ytest,
        y_pred  = predicciones,
        squared = False
       )
print("")
print(f"El error (rmse) de test es: {rmse}")

In [ ]:
init=80
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(12, 9))
real=Ytest[init:init+30]
plt.plot(real,marker='o',alpha=0.6, label="valor real")
pred=predicciones.to_numpy()
p=pred[init:init+30]
plt.plot(p,marker='*',alpha=0.6, label="valor predicción")
axes.set_title("Valor real vs prediccion", fontsize = 10, fontweight = "bold")
axes.tick_params(labelsize = 8)
axes.set_xlabel("# muestra")
axes.set_ylabel("Precio de venta")
legend = axes.legend(loc='upper center', shadow=True, fontsize='x-large')
fig.tight_layout()
plt.show()

In [ ]:
Xtest.head(1)

<h2>ACTUALIZAR formula del modelo</h2>
La formula del modelo completo es:
$$sale\_price=5.876e^5-9.219e^4 neighborhood_1+4.653e^4 neighborhood_2$$
$$+3.194e^4 neighborhood_{3}-5.805e^4 neighborhood_{4}+1.168e^5 neighborhood_{5}$$
$$-1.334e^5 neighborhood_{6}-1.898e^5 neighborhood_{7}+3.164e^4 neighborhood_{8}$$
$$-6.345e^4 neighborhood_{9}-1.978e^5 neighborhood_{10}-2.244e^5 neighborhood_{11}$$
$$-4.356e^4 neighborhood_{12}-1.792e^5 neighborhood_{13}-2.633e^4 neighborhood_{14}$$
$$+4.66e^4 neighborhood_{15}-2.396e^4 neighborhood_{16}-1.357e^5 neighborhood_{17}$$
$$-1.563e^5 neighborhood_{19}-1.776e^4 neighborhood_{20}+1.396e^5 neighborhood_{22}$$
$$-7.685e^4 neighborhood_{23}-1.267e^5 tax\_class_2+8.662e^4 tax\_class_6$$
$$+1.256e^5 tax\_class_7-2.093e^5 tax\_class_8-1.014e^5 tax\_class_9-2.803e^4 SplitZone_1$$
$$+6.44e^4 ProxCode_1+3.975e^4 ProxCode_2+4.701e^4 ProxCode_3-1.167 e^4 IrrLotCode_1$$ 
$$+56.863 lot+1066.8872 residential\_units+5.859 e^4 commercial\_units$$
$$+3779.5656 NumFloors-120.9016 LotFront-654.6348 BldgFront-104.1999 BldgDepth$$
$$-16.6075 age-6149.2409 commercial\_units*NumFloors-196.3922 lot*commercial\_units$$

Como la propiedad tiene neighborhood=6, <span style="color:skyblue">tax_class=1, SplitZone=0</span>, ProxCode=3 y<span style="color:skyblue"> IrrLoteCode=0</span> el resto de las variables dummy se consideran 0, de forma que el modelo para este caso se queda en:
$$sale\_price=5.876e^5-1.334e^5 neighborhood_{6}+4.701e^4 ProxCode_3$$ 
$$+56.863 lot+1066.8872 residential\_units+5.859 e^4 commercial\_units$$
$$+3779.5656 NumFloors-120.9016 LotFront-654.6348 BldgFront-104.1999 BldgDepth$$
$$-16.6075 age-6149.2409 commercial\_units*NumFloors-196.3922 lot*commercial\_units$$
sustituyendo:
<div style="margin:0 auto;width:400px;">
sale_price=5.876e^5-1.334e^5+4.701e^4<br/> 
+56.863*32+1066.8872 *3<span style="color:skyblue">+5.859 e^4 *0</span>+3779.5656 *3<br/>
-120.9016 *20-654.6348 *20-104.1999 *35<br/>
-16.6075 *108<span style="color:skyblue">-6149.2409 *0*3-196.3922 *32*0</span><br/>
=496617.6399
</div>

In [ ]:
predicciones.head(1)

<h3>Interpretación</h3>
El modelo de regresión lineal múltiple es capaz de explicar el 25.2% de la varianza observada en las ventas (R-squared: 0.252, Adj. R-squared: 0.252). El test F es significativo (p-value: 0.0). Por lo que hay evidencias claras de que el modelo es capaz de explicar la varianza en las ventas mejor de lo esperado por azar. Los test estadísticos para cada variable confirman que las características seleccionadas están relacionadas con el precio de venta y contribuyen al modelo.
<p>No se satisfacen las condiciones de normalidad, por lo que los intervalos de confianza estimados para los coeficientes y las predicciones no son fiables.</p>
<p>El error (rmse) de test es de 226335.74. Las predicciones del modelo final se alejan en promedio 226335.74 unidades del valor real.</p>

<h3>Conclusión</h3>
Después de realizar un análisis exhaustivo del conjunto de datos de la Casa de Brooklyn, tenemos una mejor comprensión de las características que afectan el precio de venta de una casa. Con los datos preprocesados y limpios, entrenamos un modelo predictivo para estimar el precio de venta de una casa en Brooklyn. Esta información puede ser valiosa para que los agentes inmobiliarios, los inversores y los compradores de viviendas tomen decisiones informadas.